# 04 Evaluate the Reliability Gate

This notebook selects two probability thresholds on validation data and
evaluates the frozen policy once on the held-out test subjects.

- Reliable: probability is at or above the upper threshold.
- Caution: probability lies between both thresholds.
- Unreliable: probability is at or below the lower threshold.

In [ ]:
from pathlib import Path
import os
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "apps").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the pure-pulsesure repository")


REPO_ROOT = find_repository_root(Path.cwd().resolve())
AI_SERVICE_ROOT = REPO_ROOT / "apps" / "ai-service"
sys.path.insert(0, str(AI_SERVICE_ROOT))

RAW_DIR = Path(
    os.environ.get("PURE_BUT_PPG_DIR", REPO_ROOT / "data" / "raw" / "but-ppg-2.0.0")
).expanduser().resolve()
PROCESSED_DIR = Path(
    os.environ.get("PURE_PROCESSED_DIR", REPO_ROOT / "data" / "processed" / "but-ppg-v2")
).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PURE_MODEL_DIR", REPO_ROOT / "models")).expanduser().resolve()

print("Repository:", REPO_ROOT)
print("Raw dataset:", RAW_DIR)
print("Processed dataset:", PROCESSED_DIR)
print("Model directory:", MODEL_DIR)

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

selection = json.loads((MODEL_DIR / "model_selection.json").read_text())
FEATURE_COLUMNS = selection["feature_columns"]
model_name = selection["selected_model"]
model = joblib.load(MODEL_DIR / "candidates" / f"{model_name}.joblib")
frame = pd.read_csv(PROCESSED_DIR / "features.csv")
validation = frame[frame["split"] == "validation"].copy()
test = frame[frame["split"] == "test"].copy()
validation_probability = model.predict_proba(validation[FEATURE_COLUMNS])[:, 1]
test_probability = model.predict_proba(test[FEATURE_COLUMNS])[:, 1]

In [ ]:
def gate_metrics(labels, probabilities, lower, upper):
    labels = np.asarray(labels).astype(int)
    probabilities = np.asarray(probabilities)
    reliable = probabilities >= upper
    unreliable = probabilities <= lower
    caution = ~(reliable | unreliable)
    good = labels == 1
    poor = labels == 0
    return {
        "lower_threshold": float(lower),
        "upper_threshold": float(upper),
        "false_acceptance_rate": float(np.mean(reliable[poor])),
        "false_rejection_rate": float(np.mean(unreliable[good])),
        "good_acceptance_rate": float(np.mean(reliable[good])),
        "poor_rejection_rate": float(np.mean(unreliable[poor])),
        "reliable_coverage": float(np.mean(reliable)),
        "caution_rate": float(np.mean(caution)),
        "unreliable_rate": float(np.mean(unreliable)),
    }

labels_validation = validation["label_good_quality"].to_numpy()
TARGET_FAR = 0.05
TARGET_FRR = 0.05

upper_candidates = []
for upper in np.linspace(0.50, 0.99, 100):
    metrics = gate_metrics(labels_validation, validation_probability, 0.0, upper)
    upper_candidates.append(metrics)
feasible_upper = [m for m in upper_candidates if m["false_acceptance_rate"] <= TARGET_FAR]
if feasible_upper:
    upper_threshold = max(feasible_upper, key=lambda m: m["good_acceptance_rate"])["upper_threshold"]
else:
    upper_threshold = min(upper_candidates, key=lambda m: (m["false_acceptance_rate"], -m["good_acceptance_rate"]))["upper_threshold"]

lower_candidates = []
for lower in np.linspace(0.01, min(0.49, upper_threshold - 0.01), 100):
    metrics = gate_metrics(labels_validation, validation_probability, lower, upper_threshold)
    lower_candidates.append(metrics)
feasible_lower = [m for m in lower_candidates if m["false_rejection_rate"] <= TARGET_FRR]
if feasible_lower:
    lower_threshold = max(feasible_lower, key=lambda m: m["poor_rejection_rate"])["lower_threshold"]
else:
    lower_threshold = min(lower_candidates, key=lambda m: (m["false_rejection_rate"], -m["poor_rejection_rate"]))["lower_threshold"]

validation_gate = gate_metrics(labels_validation, validation_probability, lower_threshold, upper_threshold)
validation_gate

In [ ]:
labels_test = test["label_good_quality"].to_numpy()
test_gate = gate_metrics(labels_test, test_probability, lower_threshold, upper_threshold)

accepted = test_probability >= upper_threshold
valid_hr = accepted & np.isfinite(test["estimated_bpm"]) & (test["estimated_bpm"] > 0)
hr_mae = float(np.mean(np.abs(
    test.loc[valid_hr, "estimated_bpm"] - test.loc[valid_hr, "reference_hr_bpm"]
))) if valid_hr.any() else None

test_metrics = {
    "model": model_name,
    "validation_gate": validation_gate,
    "test_gate": test_gate,
    "accepted_signal_hr_mae_bpm": hr_mae,
    "accepted_signal_hr_sample_count": int(valid_hr.sum()),
    "limitations": [
        "Thresholds are prototype validation-set decisions, not clinical standards.",
        "BUT PPG labels are record-level and binary.",
        "Caution represents model uncertainty, not a native dataset class.",
    ],
}
display(pd.Series(test_gate).to_frame("test"))
print("Accepted-signal HR MAE:", hr_mae)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(x=test_probability, hue=test["label_good_quality"], bins=25, element="step", ax=axes[0])
axes[0].axvline(lower_threshold, color="#E9C46A", linestyle="--", label="lower")
axes[0].axvline(upper_threshold, color="#147D78", linestyle="--", label="upper")
axes[0].set_title("Held-out probability distribution")
axes[0].legend()

states = np.where(test_probability >= upper_threshold, "Reliable", np.where(test_probability <= lower_threshold, "Unreliable", "Caution"))
pd.Series(states).value_counts().reindex(["Reliable", "Caution", "Unreliable"]).plot(kind="bar", ax=axes[1], color=["#35C4B5", "#E9C46A", "#E76F51"])
axes[1].set_title("Held-out quality-gate decisions")
axes[1].set_ylabel("Records")
plt.tight_layout()

In [ ]:
thresholds = {
    "model": model_name,
    "lower_threshold": float(lower_threshold),
    "upper_threshold": float(upper_threshold),
    "quality_score_definition": "round(100 * good_quality_probability)",
    "status_policy": {
        "reliable": "probability >= upper_threshold",
        "caution": "lower_threshold < probability < upper_threshold",
        "unreliable": "probability <= lower_threshold",
    },
}
MODEL_DIR.mkdir(parents=True, exist_ok=True)
(MODEL_DIR / "quality_gate_thresholds.json").write_text(json.dumps(thresholds, indent=2) + "\n")
(MODEL_DIR / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2) + "\n")
thresholds